### Data level security using Row Filters and Column Masks

####Step 1: Setup Schema for a table

In [0]:
use catalog demo;
use schema data_security;

####Step 2: Create a table

A table is defined to represent sales data accross multiple regions

In [0]:
create or replace table sales_secured(
    id INT,
    region string, 
    email string,
    revenue int
)

#### Step 3: Insert Sample Data

In [0]:
insert into sales_secured values
(1, 'UK', 'john.doe@email.com', 10000),
(2, 'US', 'mike.smith@email.com', 20000),
(3, 'UK', 'sara.jones@email.com', 30000),
(4, 'US', 'david.brown@email.com', 40000),
(5, 'UK', 'emma.wilson@email.com', 50000)

In [0]:
select * from sales_secured

#### Step 4: Define Row Filter function

The function evaluates the user's group membership and returns a condition that determines whether a row should be included in the result.

For Example:
 - Users in the UK group are allowed to view UK records
 - Users in the US group are allowed to view US records
 - Admin users are allowed to view all the records

In [0]:
create or replace function fn_filter_region(region string)
return
    is_account_group_member('admin-sg')
    or (is_account_group_member('uk-sg') and region = 'UK')
    or (is_account_group_member('us-sg') and region = 'US')

#### Step 5: Apply Row Filter to a table

The row filter function is attached to the table. Once applied, the function is evaluated automatically at query time, and only the rows thats satisfy the condition are returned

In [0]:
alter table sales_secured
set row filter fn_filter_region ON (region);

In [0]:
desc extended sales_secured

In [0]:
select * from sales_secured;

#### Step 6: Define Column Mask Function

The function evaluates the user's group membership and determines whether to return the original value or a masked version

For Example:

- Admin users can view the full email address
- Other users see a masked version of email address

In [0]:
create or replace function fn_mask_email(email string)
return
    case
        when is_account_group_member("admin-sg") then email
        else concat(substr(email,1,1), '***@', split(email,'@')[1])
    end;

#### Step 7: Apply column mask 

The masking function is attached to the email column. Once applied, the function is executed automatically whenever the column is queried

In [0]:
alter table sales_secured
alter column email set mask fn_mask_email;

In [0]:
desc extended sales_secured;

In [0]:
select * from sales_secured;